# Phase 6a-bis — Dédoublonner le SFT et mesurer la récitation

On a découvert que ton modèle SFT **récite** : 142 instructions sur la Révolution française, dont plusieurs réponses quasi identiques, et il a appris la phrase par cœur.

Le plan :
1. **mesurer** à quel point le jeu de données est redondant ;
2. **dédoublonner** ;
3. réentraîner le SFT avec **1 seule époque** ;
4. **comparer** l'ancien et le nouveau modèle avec un *taux de recopie* : la part de leurs réponses copiée mot pour mot du jeu d'entraînement.

**Kaggle :** Accelerator **GPU T4 x2**, Internet **On**. Ton token Hugging Face (Write) sera demandé : ton modèle instruct est privé, il en faut un pour le télécharger.

In [4]:
!pip -q install tokenizers safetensors huggingface_hub
import os, sys, json, math, time, random, re
from collections import Counter, defaultdict
from getpass import getpass
import numpy as np, torch, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_model, save_model
from tokenizers import Tokenizer

login(token=getpass("Token Hugging Face (Write) : "))

TRAVAIL = "/kaggle/working"
BASE, INSTRUCT = "Ezeck01/petit-gpt-fr-46m", "Ezeck01/petit-gpt-fr-46m-instruct"
for f in ["model.py", "config.json", "model.safetensors"]:
    hf_hub_download(BASE, f, local_dir=f"{TRAVAIL}/base")
for f in ["config.json", "tokenizer.json", "model.safetensors"]:
    hf_hub_download(INSTRUCT, f, local_dir=f"{TRAVAIL}/ancien")
sys.path.append(f"{TRAVAIL}/base")
from model import Config, GPT

device = "cuda"
print("GPU :", torch.cuda.get_device_name(0))

def charger(dossier):
    m = GPT(Config(**json.load(open(f"{dossier}/config.json"))))
    load_model(m, f"{dossier}/model.safetensors")
    return m.to(device).eval()

ancien = charger(f"{TRAVAIL}/ancien")                        # ton SFT actuel (2 époques, avec doublons)
tokenizer = Tokenizer.from_file(f"{TRAVAIL}/ancien/tokenizer.json")   # déjà avec <|utilisateur|> et <|assistant|>
ID_UTIL, ID_ASSIST, ID_FIN = (tokenizer.token_to_id(t) for t in ("<|utilisateur|>", "<|assistant|>", "<|endoftext|>"))
config = ancien.config

chemin = hf_hub_download("jpacifico/French-Alpaca-dataset-Instruct-110K",
                         "Acquiesce_data_110k_instructions.json", repo_type="dataset", local_dir=TRAVAIL)
donnees = json.load(open(chemin, encoding="utf-8"))
print(f"Ancien modèle chargé | {len(donnees):,} exemples")

Token Hugging Face (Write) :  ········


model.py:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  186MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  186MB            

model.safetensors: downloading bytes:           |  0.00B            

GPU : Tesla T4
Ancien modèle chargé | 110,368 exemples


## 1. Mesurer la redondance

Deux niveaux de doublons :
- **exacts** : la même réponse, au caractère près ;
- **quasi-doublons** : des réponses qui **commencent pareil**. On normalise (minuscules, ponctuation retirée) et on compare les **10 premiers mots**. C'est grossier mais efficace : c'est exactement le cas de « La Révolution française était une période de bouleversements politiques… ».

Il existe des méthodes plus fines (MinHash, similarité d'embeddings), qui détectent aussi des reformulations. On reste sur la version simple : tu peux lire et vérifier chaque groupe.

In [5]:
def normaliser(texte):
    return re.sub(r"[^\w\s]", " ", texte.lower()).split()

N_MOTS = 10
def cle(ex):
    return " ".join(normaliser(ex["output"])[:N_MOTS])

exacts = Counter(ex["output"].strip() for ex in donnees)
n_exacts = sum(n - 1 for n in exacts.values() if n > 1)

groupes = defaultdict(list)
for ex in donnees:
    groupes[cle(ex)].append(ex)
n_quasi = sum(len(g) - 1 for g in groupes.values() if len(g) > 1)

print(f"Doublons exacts      : {n_exacts:,} ({n_exacts / len(donnees):.1%})")
print(f"Quasi-doublons       : {n_quasi:,} ({n_quasi / len(donnees):.1%})  (même début sur {N_MOTS} mots)")
print("\nLes 12 débuts de réponse les plus répétés :")
for debut, g in sorted(groupes.items(), key=lambda kv: -len(kv[1]))[:12]:
    print(f"  {len(g):>4} x  « {debut} … »")

Doublons exacts      : 57,045 (51.7%)
Quasi-doublons       : 61,496 (55.7%)  (même début sur 10 mots)

Les 12 débuts de réponse les plus répétés :
   236 x  « pourquoi les plongeurs plongent ils toujours en arrière et jamais … »
   230 x  «  … »
   152 x  « la photosynthèse est le processus par lequel les plantes convertissent … »
   114 x  « le petit prince raconte l histoire d un jeune prince … »
    98 x  « la photosynthèse est le processus par lequel les plantes les … »
    96 x  « joyeux … »
    94 x  « la photosynthèse est le processus par lequel les plantes utilisent … »
    84 x  « x 5 … »
    70 x  « x 4 … »
    70 x  « oui … »
    62 x  « x 3 … »
    56 x  « la capitale de l australie est canberra … »


Regarde la liste : certains débuts sont des **formules** (« voici une liste de… ») plutôt que des copies. En dédoublonnant, on en retire aussi. Ce n'est pas un problème : le modèle n'a pas besoin de voir 300 fois la même formule pour l'apprendre.

## 2. Dédoublonner

Pour chaque groupe, on garde **un seul exemple**, choisi au hasard. On retire aussi les exemples avec du code et ceux de plus de 512 tokens, comme avant.

**Point important :** on dédoublonne **avant** de séparer entraînement et validation. Sinon, une réponse pourrait être dans le train et sa quasi-copie dans la validation. Le modèle « réussirait » la validation en récitant, et on se croirait meilleur qu'on ne l'est : c'est de la **contamination**.

In [6]:
random.seed(0)
gardes = [random.choice(g) for g in groupes.values()]

def formater(ex):
    demande = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    return demande.strip(), ex["output"].strip()

def encoder(demande, reponse=None):
    ids = [ID_UTIL] + tokenizer.encode("\n" + demande + "\n").ids + [ID_ASSIST] + tokenizer.encode("\n").ids
    if reponse is None:
        return ids
    ids_reponse = tokenizer.encode(reponse).ids + [ID_FIN]
    return ids + ids_reponse, [-100] * len(ids) + ids_reponse

exemples = []
for ex in gardes:
    demande, reponse = formater(ex)
    if not reponse or "```" in reponse or "def " in reponse:
        continue
    ids, cibles = encoder(demande, reponse)
    if len(ids) <= config.block_size:
        exemples.append((ids, cibles, ex))

random.shuffle(exemples)
val_ex, train_ex = exemples[:1000], exemples[1000:]
print(f"Avant : {len(donnees):,} exemples -> après dédoublonnage : {len(exemples):,} "
      f"({1 - len(exemples) / len(donnees):.0%} retirés)")
print(f"Train : {len(train_ex):,} | validation : {len(val_ex):,}")

Avant : 110,368 exemples -> après dédoublonnage : 48,714 (56% retirés)
Train : 47,714 | validation : 1,000


## 3. Réentraîner le SFT : 1 époque, sans doublons

On repart du **modèle de base** (pas de l'ancien SFT, sinon il garderait ce qu'il a mémorisé). Mêmes réglages qu'en 6a, mais **une seule époque**.

In [7]:
TAILLE_LOT, LR_MAX, WARMUP = 32, 2e-4, 100

# Le modèle de base, agrandi de 2 tokens comme en 6a
base = GPT(Config(**json.load(open(f"{TRAVAIL}/base/config.json"))))
load_model(base, f"{TRAVAIL}/base/model.safetensors")
etat = base.state_dict()
ancienne = etat.pop("emb_tokens.weight"); etat.pop("tete.weight")
nouvelle = torch.cat([ancienne, ancienne.mean(0, keepdim=True).repeat(2, 1)])
nouveau = GPT(config)
nouveau.load_state_dict({**etat, "emb_tokens.weight": nouvelle, "tete.weight": nouvelle})
nouveau.to(device)
del base

def faire_lot(exs):
    L = max(len(e[0]) for e in exs)
    x = torch.full((len(exs), L), ID_FIN, dtype=torch.long)
    y = torch.full((len(exs), L), -100, dtype=torch.long)
    for k, (ids, cib, _) in enumerate(exs):
        x[k, :len(ids)] = torch.tensor(ids)
        y[k, :len(cib) - 1] = torch.tensor(cib[1:])
    return x.to(device), y.to(device)

@torch.no_grad()
def perte_val(m):
    m.eval()
    total, n = 0.0, 0
    for k in range(0, len(val_ex), TAILLE_LOT):
        x, y = faire_lot(val_ex[k:k + TAILLE_LOT])
        with torch.autocast("cuda", dtype=torch.float16):
            _, perte = m(x, y)
        nb = (y != -100).sum().item()
        total += perte.item() * nb; n += nb
    m.train()
    return total / n

exs = train_ex[:]
random.shuffle(exs)
paquets = [sorted(exs[k:k + 3200], key=lambda e: len(e[0])) for k in range(0, len(exs), 3200)]
lots = [p[k:k + TAILLE_LOT] for p in paquets for k in range(0, len(p), TAILLE_LOT)]
random.shuffle(lots)
ETAPES = len(lots)

def taux(e):
    if e < WARMUP:
        return LR_MAX * (e + 1) / WARMUP
    return LR_MAX * 0.5 * (1 + math.cos(math.pi * (e - WARMUP) / (ETAPES - WARMUP)))

print(f"Perte val AVANT : {perte_val(nouveau):.3f}")
optimiseur = torch.optim.AdamW(nouveau.parameters(), lr=LR_MAX, betas=(0.9, 0.95), weight_decay=0.1, fused=True)
scaler = torch.amp.GradScaler()
nouveau.train()
t0 = time.time()
for etape, exs_lot in enumerate(lots):
    for g in optimiseur.param_groups:
        g["lr"] = taux(etape)
    x, y = faire_lot(exs_lot)
    with torch.autocast("cuda", dtype=torch.float16):
        _, perte = nouveau(x, y)
    scaler.scale(perte).backward()
    scaler.unscale_(optimiseur)
    torch.nn.utils.clip_grad_norm_(nouveau.parameters(), 1.0)
    scaler.step(optimiseur); scaler.update()
    optimiseur.zero_grad(set_to_none=True)
    if etape % 200 == 0:
        reste = (ETAPES - etape) * (time.time() - t0) / max(1, etape)
        print(f"étape {etape:>5}/{ETAPES} | perte {perte.item():.3f} | reste ~{reste / 60:.0f} min")
nouveau.eval()
print(f"\nPerte val APRÈS : {perte_val(nouveau):.3f}")

Perte val AVANT : 2.972
étape     0/1492 | perte 3.020 | reste ~16 min
étape   200/1492 | perte 2.366 | reste ~2 min
étape   400/1492 | perte 2.440 | reste ~2 min
étape   600/1492 | perte 2.191 | reste ~1 min
étape   800/1492 | perte 1.747 | reste ~1 min
étape  1000/1492 | perte 1.924 | reste ~1 min
étape  1200/1492 | perte 2.076 | reste ~0 min
étape  1400/1492 | perte 1.713 | reste ~0 min

Perte val APRÈS : 1.883


⚠️ Ne compare pas cette perte de validation avec le 1,305 de la 6a : ce n'est pas le même jeu de validation. Et l'ancien était **contaminé** (des quasi-copies de ses réponses de validation étaient dans son train), donc probablement trop optimiste. La vraie comparaison, c'est la suite.

## 4. Le taux de recopie

Pour chaque réponse générée, on découpe le texte en **séquences de 8 mots consécutifs** (des 8-grammes, comme ton modèle N = 8 de la phase 0 !). Puis on regarde quelle part de ces 8-grammes existe **telle quelle** dans les réponses du jeu d'entraînement.

8 mots identiques d'affilée, c'est très rarement une coïncidence : c'est de la copie.

On teste sur deux types de questions :
- **des questions vues** : 100 instructions tirées du jeu de données. Les deux modèles les ont vues (ou une quasi-copie) ;
- **des questions nouvelles** : écrites ici, absentes des données.

In [8]:
N = 8
def ngrammes(texte):
    mots = normaliser(texte)
    return [tuple(mots[i:i + N]) for i in range(len(mots) - N + 1)]

print("Indexation des 8-grammes du jeu d'entraînement…")
index = set()
for ex in donnees:
    index.update(ngrammes(ex["output"]))
print(f"{len(index):,} 8-grammes différents")

@torch.no_grad()
def repondre(m, question, n=100):
    ids = encoder(question)
    x = torch.tensor([ids], device=device)
    for _ in range(n):
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = m(x[:, -config.block_size:])
        suivant = logits[0, -1].float().argmax()           # décodage glouton : le plus probable
        if suivant.item() == ID_FIN:
            break
        x = torch.cat([x, suivant.view(1, 1)], dim=1)
    return tokenizer.decode(x[0, len(ids):].tolist()).strip()

def taux_recopie(reponses):
    total = copies = 0
    for r in reponses:
        g = ngrammes(r)
        total += len(g)
        copies += sum(x in index for x in g)
    return copies / max(1, total)

random.seed(1)
vues = [formater(ex)[0] for ex in random.sample(donnees, 100)]
nouvelles = [
    "Explique ce qu'est un volcan.", "Quel est le contraire du mot « rapide » ?",
    "Résume l'histoire de la Révolution française en deux phrases.",
    "Pourquoi le ciel est-il bleu ?", "Donne trois idées de prénoms pour un chat.",
    "Décris une journée de pluie à Ouagadougou.", "Qu'est-ce qu'un nombre premier ?",
    "Écris une phrase avec les mots « soleil » et « mer ».", "Comment faire un bon café ?",
    "Pourquoi faut-il dormir ?", "Quels sont les avantages du vélo en ville ?",
    "Donne un conseil à un étudiant qui commence un master.", "Qu'est-ce que la photosynthèse ?",
    "Décris le goût d'une mangue.", "Comment s'appelle la capitale du Canada ?",
    "Explique la différence entre un fleuve et une rivière.", "Pourquoi les feuilles tombent-elles en automne ?",
    "Écris un slogan pour une boulangerie.", "Qu'est-ce qu'une langue peu dotée ?",
    "Donne deux synonymes du mot « content ».",
]

resultats = {}
for nom, m in [("ancien (2 époques, doublons)", ancien), ("nouveau (1 époque, dédoublonné)", nouveau)]:
    t0 = time.time()
    rv = [repondre(m, q) for q in vues]
    rn = [repondre(m, q) for q in nouvelles]
    resultats[nom] = (rv, rn)
    print(f"{nom:<34} | recopie questions vues : {taux_recopie(rv):.0%} | "
          f"questions nouvelles : {taux_recopie(rn):.0%}  ({time.time() - t0:.0f} s)")

Indexation des 8-grammes du jeu d'entraînement…
2,018,020 8-grammes différents
ancien (2 époques, doublons)       | recopie questions vues : 27% | questions nouvelles : 26%  (48 s)
nouveau (1 époque, dédoublonné)    | recopie questions vues : 8% | questions nouvelles : 3%  (49 s)


## 5. Côte à côte

Les réponses des deux modèles aux questions nouvelles. On utilise le décodage **glouton** (toujours le token le plus probable) : pas de hasard, donc la comparaison est équitable.

In [9]:
(_, rn_ancien), (_, rn_nouveau) = resultats.values()
for q, a, b in zip(nouvelles, rn_ancien, rn_nouveau):
    print("❓", q)
    print("   ANCIEN  :", a.replace("\n", " / ")[:250])
    print("   NOUVEAU :", b.replace("\n", " / ")[:250])
    print()

❓ Explique ce qu'est un volcan.
   ANCIEN  : Un volcan est un volcan qui se trouve à la surface de la Terre. Il est composé de lave, de cendres et de gaz, qui sont généralement associés à un ciel ensoleillé.
   NOUVEAU : Un volcan est un volcan en forme de cratère, qui se trouve à environ 7 km de la Terre. Il est situé à environ 8 km de la Terre, et est le plus haut sommet du monde.

❓ Quel est le contraire du mot « rapide » ?
   ANCIEN  : Le contraire du motquité signifie que quelque chose peut être fait sans l'aide de quelque chose.
   NOUVEAU : Le mot " rapide" est un nom commun.

❓ Résume l'histoire de la Révolution française en deux phrases.
   ANCIEN  : La Révolution française était une période de bouleversements politiques et sociaux en France, qui a duré de 1789 à 1799. Elle a abouti à la chute de la monarchie, à l'établissement de la Première République et à des changements radicaux dans la socié
   NOUVEAU : La Révolution française a été un événement majeur de la Révolution f

## 6. Publier la nouvelle version

On met à jour le **même dépôt** `petit-gpt-fr-46m-instruct`. Hugging Face garde l'historique : l'ancienne version reste consultable dans l'onglet *Files → History*.

Ne lance cette cellule que si le nouveau modèle te semble **meilleur ou équivalent**. Sinon, garde l'ancien.

In [10]:
from huggingface_hub import HfApi
from dataclasses import asdict
import shutil

(rv_a, rn_a), (rv_n, rn_n) = resultats.values()
SORTIE = f"{TRAVAIL}/instruct_v2"
os.makedirs(SORTIE, exist_ok=True)
save_model(nouveau.cpu(), f"{SORTIE}/model.safetensors"); nouveau.to(device)
json.dump(asdict(config), open(f"{SORTIE}/config.json", "w"), indent=2)
tokenizer.save(f"{SORTIE}/tokenizer.json")
shutil.copy(f"{TRAVAIL}/base/model.py", f"{SORTIE}/model.py")

fiche = f"""---
language: fr
license: apache-2.0
library_name: pytorch
tags: [text-generation, gpt, from-scratch, french, sft]
base_model: {BASE}
datasets: [jpacifico/French-Alpaca-dataset-Instruct-110K]
---

# Petit GPT français — version instruct (SFT), v2 dédoublonnée

Version ajustée par SFT de [{BASE}](https://huggingface.co/{BASE}), un GPT de 46 M de paramètres entraîné de zéro sur Wikipédia en français.

## Données

French-Alpaca-dataset-Instruct-110K (Jonathan Pacifico, 2024), réponses générées par GPT-3.5-turbo.

La v1 récitait des réponses entières (ex. 142 instructions sur la Révolution française, plusieurs réponses quasi identiques).
La v2 **dédoublonne** avant de séparer train et validation : un seul exemple par début de réponse identique (10 premiers mots normalisés).
{len(donnees):,} -> {len(exemples):,} exemples. Exemples contenant du code ou dépassant 512 tokens écartés.

## Entraînement

1 époque (contre 2 en v1), lots de {TAILLE_LOT} exemples, AdamW, taux d'apprentissage {LR_MAX}, warmup puis cosinus.
Deux tokens spéciaux (`<|utilisateur|>`, `<|assistant|>`), perte calculée uniquement sur les réponses.

## Taux de recopie

Part des 8-grammes de mots des réponses générées (décodage glouton) présents tels quels dans les réponses du jeu d'entraînement.

| | Questions vues ({len(vues)}) | Questions nouvelles ({len(nouvelles)}) |
|---|---|---|
| v1 (2 époques, avec doublons) | {taux_recopie(rv_a):.0%} | {taux_recopie(rn_a):.0%} |
| v2 (1 époque, dédoublonné) | {taux_recopie(rv_n):.0%} | {taux_recopie(rn_n):.0%} |

## Format

```
<|utilisateur|>
Ta question
<|assistant|>
```
La réponse se termine par `<|endoftext|>`.

## Limites

46 M de paramètres : connaissances très limitées, le modèle invente souvent. Projet d'apprentissage, pas un assistant fiable.
"""
open(f"{SORTIE}/README.md", "w", encoding="utf-8").write(fiche)

HfApi().upload_folder(folder_path=SORTIE, repo_id=INSTRUCT, commit_message="v2 : SFT dédoublonné, 1 époque")
print(f"Mis à jour : https://huggingface.co/{INSTRUCT}")

Mis à jour : https://huggingface.co/Ezeck01/petit-gpt-fr-46m-instruct
